# Stream 2 — master Colab workflow
Run top-to-bottom with T4 GPU. Upload repository ZIP and existing `cda_ur_full.zip` first. **Training can take hours.** Checkpoints are saved/downloaded after each model. Historical controls should not be mixed with final checkpoints.


In [ ]:
from pathlib import Path
import os, zipfile, shutil, torch
from google.colab import files
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Enable GPU in Runtime > Change runtime type'
print('Upload (1) latest repository ZIP containing data/cda/pairs_hi.jsonl, and (2) cda_ur_full.zip')
files.upload()
repo=Path('/content/nlp-gender-bias-urdu-main')
if not (repo/'src/disco/control.py').exists():
    for z in Path('/content').glob('*.zip'):
        if 'cda_ur_full' in z.name: continue
        with zipfile.ZipFile(z) as f:
            if any(x.endswith('src/disco/control.py') for x in f.namelist()):
                f.extractall('/content'); break
assert (repo/'src/disco/control.py').exists(), 'Repository ZIP missing or extracted under a different folder'
assert (repo/'data/cda/pairs_hi.jsonl').exists(), 'Download the LATEST repository ZIP from main (Hindi data missing)'
for z in Path('/content').glob('cda_ur_full.zip'):
    with zipfile.ZipFile(z) as f: f.extractall('/content')
assert (Path('/content/cda_ur_full')/'config.json').exists(), 'Urdu checkpoint not found; inspect ZIP layout'
os.chdir(repo)
print('READY:',repo)


GPU: Tesla T4
Upload (1) latest repository ZIP containing data/cda/pairs_hi.jsonl, and (2) cda_ur_full.zip


Saving nlp-gender-bias-urdu-main.zip to nlp-gender-bias-urdu-main.zip
Saving cda_ur_full.zip to cda_ur_full.zip
READY: /content/nlp-gender-bias-urdu-main


## Train missing English and English+Urdu models
Each cell skips training if a valid model checkpoint exists, and downloads a ZIP immediately after training. Do not close Colab before each download finishes.


### cda_en_full

In [ ]:
from pathlib import Path
if not (Path("/content/cda_en_full") / "config.json").exists():

    import json
    import torch
    from pathlib import Path
    from datasets import Dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForMaskedLM,
        DataCollatorForLanguageModeling,
        Trainer,
        TrainingArguments,
    )

    MODEL = "xlm-roberta-base"
    OUTPUT = "/content/cda_en_full"

    # Load all English CDA pairs
    sentences = []

    with open("data/cda/en_pairs.jsonl", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            sentences.extend([row["en"], row["en_cf"]])

    print("Training sentences:", len(sentences))

    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForMaskedLM.from_pretrained(MODEL)

    dataset = Dataset.from_dict({"text": sentences})

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=128,
            return_special_tokens_mask=True,
        )

    dataset = dataset.map(
        tokenize,
        batched=True,
        remove_columns=["text"],
    )

    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm_probability=0.15,
    )

    args = TrainingArguments(
        output_dir=OUTPUT,
        max_steps=500,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        logging_steps=25,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        fp16=True,
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=collator,
        processing_class=tokenizer,
    )

    trainer.train()

    model.save_pretrained(OUTPUT)
    tokenizer.save_pretrained(OUTPUT)

    print("FULL CDA-EN TRAINING COMPLETED!")
    print("Saved model:", OUTPUT)

else:
    print("Skipping existing checkpoint: cda_en_full")

import shutil
from google.colab import files
archive=shutil.make_archive('/content/cda_en_full', 'zip', '/content', "cda_en_full")
print('BACKUP:',archive)
files.download(archive)


Training sentences: 19944


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

... (10 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FULL CDA-EN TRAINING COMPLETED!
Saved model: /content/cda_en_full
BACKUP: /content/cda_en_full.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### cda_en_ur_full

In [ ]:
from pathlib import Path
if not (Path("/content/cda_en_ur_full") / "config.json").exists():

    import os, sys, json, re, random, gc, subprocess
    from pathlib import Path
    import torch
    from datasets import Dataset
    from transformers import (
        AutoTokenizer, AutoModelForMaskedLM,
        DataCollatorForLanguageModeling,
        Trainer, TrainingArguments
    )

    os.chdir("/content/nlp-gender-bias-urdu-main")

    BASE = "xlm-roberta-base"
    OUTPUT = "/content/cda_en_ur_full"
    SEED = 42
    random.seed(SEED)
    torch.manual_seed(SEED)

    assert torch.cuda.is_available(), "Enable T4 GPU first!"

    # Load English pairs
    with open("data/cda/en_pairs.jsonl", encoding="utf-8") as f:
        en_rows = [json.loads(x) for x in f if x.strip()]

    # Load Urdu pairs
    with open("data/cda/pairs_ur.jsonl", encoding="utf-8") as f:
        ur_rows = [json.loads(x) for x in f if x.strip()]

    def mostly_latin(text):
        latin = len(re.findall(r"[A-Za-z]", text))
        arabic = len(re.findall(r"[\u0600-\u06FF]", text))
        return latin > arabic

    # Filter Urdu translations
    ur_valid = [
        r for r in ur_rows
        if r["ur"].strip()
        and r["ur_cf"].strip()
        and not mostly_latin(r["ur"])
        and not mostly_latin(r["ur_cf"])
        and r["ur"].strip() != r["ur_cf"].strip()
    ]

    # Balance languages by number of pairs
    n = min(len(en_rows), len(ur_valid))
    rng = random.Random(SEED)
    en_sample = rng.sample(en_rows, n)
    ur_sample = rng.sample(ur_valid, n)

    sentences = []

    for r in en_sample:
        sentences.extend([r["en"], r["en_cf"]])

    for r in ur_sample:
        sentences.extend([r["ur"], r["ur_cf"]])

    rng.shuffle(sentences)

    print("English pairs:", n)
    print("Urdu pairs:", n)
    print("Total sentences:", len(sentences))

    # Tokenize
    tokenizer = AutoTokenizer.from_pretrained(BASE)
    model = AutoModelForMaskedLM.from_pretrained(BASE)

    dataset = Dataset.from_dict({"text": sentences})

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=128,
            return_special_tokens_mask=True
        )

    dataset = dataset.map(
        tokenize,
        batched=True,
        remove_columns=["text"]
    )

    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    # Train
    args = TrainingArguments(
        output_dir=OUTPUT,
        max_steps=500,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        logging_steps=25,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        fp16=True,
        seed=SEED
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=collator,
        processing_class=tokenizer
    )

    print("\nSTARTING COMBINED CDA TRAINING...")
    trainer.train()

    model.save_pretrained(OUTPUT)
    tokenizer.save_pretrained(OUTPUT)

    Path(OUTPUT, "experiment.json").write_text(
        json.dumps({
            "method": "CDA-EN+UR",
            "base_model": BASE,
            "seed": SEED,
            "english_pairs": n,
            "urdu_pairs": n,
            "total_sentences": len(sentences),
            "max_steps": 500,
            "learning_rate": 2e-5,
            "urdu_identical_pairs_excluded": True
        }, indent=2),
        encoding="utf-8"
    )

    print("\nCOMBINED CDA TRAINING COMPLETE!")

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    # Evaluate both languages
    for lang, templates, tag in [
        ("en", None, "cda_en_ur"),
        ("ur", "data/templates/v2", "cda_en_ur_v2")
    ]:
        cmd = [
            sys.executable, "-m", "src.disco.run",
            "--lang", lang,
            "--model", OUTPUT,
            "--tokenizer", BASE,
            "--tag", tag
        ]

        if templates:
            cmd += ["--templates_dir", templates]

        print(f"\nEvaluating {lang}...", flush=True)
        subprocess.run(cmd, check=True)

    # Print scores
    print("\n===== COMBINED CDA RESULTS =====")

    for lang, tag, baseline in [
        ("en", "cda_en_ur", 0.773),
        ("ur", "cda_en_ur_v2", 0.877)
    ]:
        path = Path(
            f"results/disco_{lang}_cda_en_ur_full_{tag}.json"
        )

        if path.exists():
            result = json.loads(path.read_text(encoding="utf-8"))
            score = result["score"]
            print(
                f"{lang.upper()}: baseline={baseline:.4f}, "
                f"combined={score:.4f}, "
                f"change={score-baseline:+.4f}"
            )
        else:
            print("Result file not found:", path)

    print("\nDONE!")
    print("Saved model:", OUTPUT)

else:
    print("Skipping existing checkpoint: cda_en_ur_full")

import shutil
from google.colab import files
archive=shutil.make_archive('/content/cda_en_ur_full', 'zip', '/content', "cda_en_ur_full")
print('BACKUP:',archive)
files.download(archive)


English pairs: 7840
Urdu pairs: 7840
Total sentences: 31360


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/31360 [00:00<?, ? examples/s]


STARTING COMBINED CDA TRAINING...


Step,Training Loss
25,4.410585
50,4.729166
75,4.489449
100,4.477367
125,4.358386
150,4.373771
175,4.403350
200,4.352565
225,4.347072
250,4.159716


... (5 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


COMBINED CDA TRAINING COMPLETE!

Evaluating en...

Evaluating ur...

===== COMBINED CDA RESULTS =====
EN: baseline=0.7730, combined=0.6429, change=-0.1301
UR: baseline=0.8770, combined=0.6990, change=-0.1780

DONE!
Saved model: /content/cda_en_ur_full
BACKUP: /content/cda_en_ur_full.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Train Hindi-only and Hindi+Urdu (excluding English)
Hindi data must exist in latest repo. Hindi pairs that are mostly Latin are filtered, as requested. These are *new* experiments, not part of the original 3-model run.


In [ ]:
import json,re,random,gc,shutil
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from google.colab import files

def load_pairs(filename,a,b):
    rows=[json.loads(line) for line in open(filename,encoding='utf-8') if line.strip()]
    def mostly_latin(t):
        return len(re.findall(r'[A-Za-z]',t))>len(re.findall(r'[\u0600-\u06FF\u0900-\u097F]',t))
    valid=[r for r in rows if r[a].strip() and r[b].strip() and r[a].strip()!=r[b].strip() and not mostly_latin(r[a]) and not mostly_latin(r[b])]
    print(filename,'source pairs',len(rows),'valid pairs',len(valid))
    return valid
hi=load_pairs('data/cda/pairs_hi.jsonl','hi','hi_cf')
ur=load_pairs('data/cda/pairs_ur.jsonl','ur','ur_cf')
assert hi and ur

def train_new(name, sentences, pair_counts):
    out=Path('/content')/name
    if (out/'config.json').exists():
        print('Already trained:',name)
    else:
        random.Random(42).shuffle(sentences)
        tok=AutoTokenizer.from_pretrained('xlm-roberta-base')
        model=AutoModelForMaskedLM.from_pretrained('xlm-roberta-base')
        ds=Dataset.from_dict({'text':sentences}).map(lambda b:tok(b['text'],truncation=True,max_length=128,return_special_tokens_mask=True),batched=True,remove_columns=['text'])
        args=TrainingArguments(output_dir=str(out),max_steps=500,per_device_train_batch_size=4,gradient_accumulation_steps=2,learning_rate=2e-5,logging_steps=25,save_strategy='no',report_to='none',fp16=True,seed=42)
        trainer=Trainer(model=model,args=args,train_dataset=ds,data_collator=DataCollatorForLanguageModeling(tokenizer=tok,mlm_probability=0.15),processing_class=tok)
        trainer.train();model.save_pretrained(out);tok.save_pretrained(out)
        (out/'experiment.json').write_text(json.dumps({'setup':name,'pair_counts':pair_counts,'training_sentences':len(sentences),'max_steps':500,'learning_rate':2e-5,'batch_size':4,'gradient_accumulation':2,'seed':42},indent=2))
        del trainer,model,tok,ds;gc.collect();torch.cuda.empty_cache()
    archive=shutil.make_archive(str(out),'zip','/content',name)
    print('DOWNLOAD BACKUP NOW:',archive)
    files.download(archive)

hi_sentences=[text for r in hi for text in (r['hi'],r['hi_cf'])]
train_new('cda_hi_full',hi_sentences,{'hi':len(hi)})


data/cda/pairs_hi.jsonl source pairs 9972 valid pairs 6990
data/cda/pairs_ur.jsonl source pairs 9972 valid pairs 7840


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/13980 [00:00<?, ? examples/s]

Step,Training Loss
25,4.653451
50,3.948806
75,4.456382
100,4.145953
125,4.054482
150,3.991382
175,3.747171
200,3.735370
225,4.131663
250,3.421118


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DOWNLOAD BACKUP NOW: /content/cda_hi_full.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
n=min(len(hi),len(ur))
rng=random.Random(42)
hs=rng.sample(hi,n); us=rng.sample(ur,n)
hi_ur_sentences=[text for r in hs for text in (r['hi'],r['hi_cf'])]+[text for r in us for text in (r['ur'],r['ur_cf'])]
train_new('cda_hi_ur_full',hi_ur_sentences,{'hi':n,'ur':n})


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/27960 [00:00<?, ? examples/s]

Step,Training Loss
25,4.361720
50,4.621017
75,4.276439
100,4.346879
125,4.256339
150,4.605298
175,4.120050
200,4.508329
225,3.989493
250,3.927042


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DOWNLOAD BACKUP NOW: /content/cda_hi_ur_full.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Run DisCo and matched controls
Runs official repository scripts. **Confirm the control CLI supports `--seed` and `--templates_dir` before running**; if not, adapt to repo version. This section may take hours. Save a ZIP of results at end.


In [ ]:
import subprocess,sys
from pathlib import Path
models=['cda_en_full','cda_ur_full','cda_en_ur_full','cda_hi_full','cda_hi_ur_full']
for model in models:
    assert (Path('/content')/model/'config.json').exists(),f'Missing {model}'
print('Model folders verified:',models)
print('CONTROL HELP:')
subprocess.run([sys.executable,'-m','src.disco.control','--help'],check=True)
print('DISCO HELP:')
subprocess.run([sys.executable,'-m','src.disco.run','--help'],check=True)


Model folders verified: ['cda_en_full', 'cda_ur_full', 'cda_en_ur_full', 'cda_hi_full', 'cda_hi_ur_full']
CONTROL HELP:
DISCO HELP:


CompletedProcess(args=['/usr/bin/python3', '-m', 'src.disco.run', '--help'], returncode=0)

In [ ]:
import subprocess,sys
from pathlib import Path
for name in models:
    for lang in (['en','ur'] if name not in ('cda_hi_full','cda_hi_ur_full') else ['ur']):
        path=str(Path('/content')/name)
        cmd=[sys.executable,'-m','src.disco.run','--lang',lang,'--model',path,'--tokenizer','xlm-roberta-base','--tag','stream2_final']
        if lang=='ur':cmd+=['--templates_dir','data/templates/v2']
        print('RUNNING DISCO:',name,lang,flush=True)
        subprocess.run(cmd,check=True)
        for seed in range(5):
            cmd=[sys.executable,'-m','src.disco.control','--lang',lang,'--model',path,'--seed',str(seed)]
            if lang=='ur':cmd+=['--templates_dir','data/templates/v2']
            print('CONTROL:',name,lang,seed,flush=True)
            subprocess.run(cmd,check=True)


RUNNING DISCO: cda_en_full en
CONTROL: cda_en_full en 0
CONTROL: cda_en_full en 1
CONTROL: cda_en_full en 2
CONTROL: cda_en_full en 3
CONTROL: cda_en_full en 4
RUNNING DISCO: cda_en_full ur
CONTROL: cda_en_full ur 0
CONTROL: cda_en_full ur 1
CONTROL: cda_en_full ur 2
CONTROL: cda_en_full ur 3
CONTROL: cda_en_full ur 4
RUNNING DISCO: cda_ur_full en
CONTROL: cda_ur_full en 0
CONTROL: cda_ur_full en 1
CONTROL: cda_ur_full en 2
CONTROL: cda_ur_full en 3
CONTROL: cda_ur_full en 4
RUNNING DISCO: cda_ur_full ur
CONTROL: cda_ur_full ur 0
CONTROL: cda_ur_full ur 1
CONTROL: cda_ur_full ur 2
CONTROL: cda_ur_full ur 3
CONTROL: cda_ur_full ur 4
RUNNING DISCO: cda_en_ur_full en
CONTROL: cda_en_ur_full en 0
CONTROL: cda_en_ur_full en 1
CONTROL: cda_en_ur_full en 2
CONTROL: cda_en_ur_full en 3
CONTROL: cda_en_ur_full en 4
RUNNING DISCO: cda_en_ur_full ur
CONTROL: cda_en_ur_full ur 0
CONTROL: cda_en_ur_full ur 1
CONTROL: cda_en_ur_full ur 2
CONTROL: cda_en_ur_full ur 3
CONTROL: cda_en_ur_full ur 4
RUNN

## Self-Debiasing controls
The repository `control.py` normally loads a standard masked LM and does **not** automatically invoke the custom `SelfDebiasModel.predict` method. Therefore running `control.py --model xlm-roberta-base` is a baseline control, **not a Self-Debiasing control**. Implement the matching adapter using the actual repository source before claiming this requirement is complete.


## Save all results


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/stream2_final_results','zip',str(repo),'results')
print('RESULTS BACKUP:',archive)
files.download(archive)


RESULTS BACKUP: /content/stream2_final_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path

for name in ["cda_en_full", "cda_ur_full", "cda_en_ur_full"]:
    path = Path("/content") / name
    print(name, "READY" if (path / "config.json").exists() else "MISSING")

cda_en_full READY
cda_ur_full READY
cda_en_ur_full READY


In [ ]:
import torch
print("Runtime working!")
print("GPU available:", torch.cuda.is_available())

Runtime working!
GPU available: False


In [ ]:
from pathlib import Path

print("CHECKING FILES\n")

for name in ["cda_en_full", "cda_ur_full", "cda_en_ur_full"]:
    p = Path("/content") / name
    print(name, "READY" if (p / "config.json").exists() else "NOT IN RUNTIME")

repos = list(Path("/content").glob("nlp-gender-bias-urdu*"))

for p in repos:
    if p.is_dir():
        print("\nRepository:", p)
        print("Hindi data:", (p / "data/cda/pairs_hi.jsonl").exists())
        print("Self-Debiasing:", (p / "src/disco/self_debias.py").exists())

CHECKING FILES

cda_en_full NOT IN RUNTIME
cda_ur_full NOT IN RUNTIME
cda_en_ur_full NOT IN RUNTIME
